In [1]:
!pip install -q git+https://github.com/huggingface/transformers accelerate qwen-vl-utils==0.0.8 bitsandbytes sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip install jiwer

In [3]:
import torch
print(torch.cuda.is_available())

True


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os

base_path = "/content/drive/MyDrive/dataset"

def load_dataset(base_path):
    groups = ["degraded", "dense_text", "handwritten", "printed", "receipts", "scene_text"]
    data = []

    for group in groups:
        img_dir = os.path.join(base_path, group, "images")
        gt_dir = os.path.join(base_path, group, "ground_truth")

        for file in sorted(os.listdir(img_dir)):
            if file.endswith(".png"):
                data.append({
                    "image": os.path.join(img_dir, file),
                    "label": os.path.join(gt_dir, file.replace(".png", ".txt")),
                    "group": group
                })

    return data

data = load_dataset(base_path)

print("Samples:", len(data))

Samples: 36


In [6]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

# Set min_pixels and max_pixels to control image resizing for memory optimization
min_pixels = 256 * 28 * 28  # Example value from Qwen-VL docs
max_pixels = 1280 * 28 * 28 # Example value from Qwen-VL docs
processor = AutoProcessor.from_pretrained(model_name, min_pixels=min_pixels, max_pixels=max_pixels)

# Define quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config
)

model.eval()

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Model loaded


In [7]:
from PIL import Image

sample = data[0]
image = Image.open(sample["image"]).convert("RGB")

print(sample["image"])

/content/drive/MyDrive/dataset/degraded/images/degraded_000.png


In [8]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Extract all text exactly as it appears."}
        ]
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

In [9]:
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt"
).to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False
)

generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, output)
]
pred = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print("\nPREDICTED:\n")
print(pred)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



PREDICTED:

R&D QUALITY IMPROVEMENT
SUGGESTION/SOLUTION FORM

Name/Phone Ext.: M. Hamann, P. Harper, P. Martinez Date: 9/3/92
Supervisor/Manager: J. S. Wigand R&D Group: Licensee

Suggestion: Discontinue coal retention analyses on licensee submitted product samples. (Note: Coal Retention testing is not performed by most licensees. Other SW physical measurements as ends stability and inspection for soft spots in cigarettes are thought to be sufficient measures to assure cigarette physical integrity. The proposed action will increase laboratory productivity.)

Suggested Solution(s): Delete coal retention from the list of standard analyses performed on licensee submitted product samples. Special requests for coal retention testing could still be submitted on an exception basis.

Have you contacted your Manager/Supervisor? Yes No
Manager Comments: Manager, please contact suggester and forward comments to the Quality Council.
597005708


In [10]:
!apt-get install tesseract-ocr -q
!pip install pytesseract -q

import pytesseract
import cv2

def extract_text_tesseract_receipts(image_path):
    img = cv2.imread(str(image_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return pytesseract.image_to_string(gray, lang="eng", config="--psm 6")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [11]:
sample_receipt = [item for item in data if item["group"] == "receipts"][0]
print("Path:", sample_receipt["image"])

img = cv2.imread(sample_receipt["image"])
print("Image loaded:", img is not None)

if img is not None:
    print("Shape:", img.shape)
    pred = extract_text_tesseract_receipts(sample_receipt["image"])
    print("Prediction:\n", pred)

    with open(sample_receipt["label"]) as f:
        print("Ground truth:\n", f.read())

Path: /content/drive/MyDrive/dataset/receipts/images/receipt_000.png
Image loaded: True
Shape: (3508, 2481, 3)
Prediction:
 Invoice no: 79587541
Date of issue: 08/02/2014
Seller: Client:
Baker-Wagner Horton, Gillespie and Moore
909 Duarte Drive Apt. 874 711 Richard Cove Suite 885
Joelfort, NH 24372 East Mindy, WY 58533
Tax Id: 971-99-0928 Tax Id: 965-80-2789
IBAN: GB24KU0P15238222815223
ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
1. Bottle tops lid reusable 1,00 each 9,99 9,99 10% 10,99
vacuume Sealer stopper flip cap
cork wine beer
SUMMARY
VAT [%] Net worth VAT Gross worth
10% 9,99 1,00 10,99
Total $ 9,99 $ 1,00 $ 10,99

Ground truth:
 Invoice no: 79587541 Date of issue: 08/02/2014 Seller: Client: Baker-Wagner Horton, Gillespie and Moore. 909 Duarte Drive Apt. 874 711 Richard Cove Suite 885. Joelfort, NH 24372 East Mindy, WY 58533 Tax Id: 971-99-0928 Tax Id: 965-80-2789 IBAN: GB24KUOP15238222815223 ITEMS UM No. Description Qty Net price Net worth VAT [%] Gros

In [12]:
from jiwer import cer, wer
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

pred = extract_text_tesseract_receipts(sample_receipt["image"])
with open(sample_receipt["label"]) as f:
    gt = f.read()

print("CER:", cer(normalize_text(gt), normalize_text(pred)))
print("WER:", wer(normalize_text(gt), normalize_text(pred)))
print("\nNORMALIZED PRED:\n", normalize_text(pred))
print("\nNORMALIZED GT:\n", normalize_text(gt))

CER: 0.035343035343035345
WER: 0.07142857142857142

NORMALIZED PRED:
 invoice no 79587541 date of issue 08022014 seller client bakerwagner horton gillespie and moore 909 duarte drive apt 874 711 richard cove suite 885 joelfort nh 24372 east mindy wy 58533 tax id 971990928 tax id 965802789 iban gb24ku0p15238222815223 items no description qty um net price net worth vat gross worth 1 bottle tops lid reusable 100 each 999 999 10 1099 vacuume sealer stopper flip cap cork wine beer summary vat net worth vat gross worth 10 999 100 1099 total 999 100 1099

NORMALIZED GT:
 invoice no 79587541 date of issue 08022014 seller client bakerwagner horton gillespie and moore 909 duarte drive apt 874 711 richard cove suite 885 joelfort nh 24372 east mindy wy 58533 tax id 971990928 tax id 965802789 iban gb24kuop15238222815223 items um no description qty net price net worth vat gross worth bottle tops lid reusable 100 each 999 999 10 1099 vacuume sealer stopper flip cap cork wine beer summary vat vat net 

In [13]:
for item in data:
    if item["group"] != "receipts":
        continue

    pred = extract_text_tesseract_receipts(item["image"])
    with open(item["label"]) as f:
        gt = f.read()

    from jiwer import cer, wer
    c = cer(normalize_text(gt), normalize_text(pred))
    w = wer(normalize_text(gt), normalize_text(pred))
    acc = max(0.0, (1.0 - ((c + w) / 2.0)) * 100)
    print(f"{item['image'].split('/')[-1]} → acc: {round(acc,2)}, cer: {round(c,3)}, wer: {round(w,3)}")

receipt_000.png → acc: 94.66, cer: 0.035, wer: 0.071
receipt_001.png → acc: 93.91, cer: 0.043, wer: 0.079
receipt_002.png → acc: 82.92, cer: 0.123, wer: 0.218
receipt_003.png → acc: 82.59, cer: 0.129, wer: 0.22
receipt_004.png → acc: 86.84, cer: 0.086, wer: 0.177
receipt_005.png → acc: 84.91, cer: 0.11, wer: 0.192


In [15]:
predictions = []
for item in data:
    torch.cuda.empty_cache()

    if item["group"] == "receipts":
        pred = extract_text_tesseract_receipts(item["image"])
        print(f"TESSERACT: {item['image'].split('/')[-1]} → {pred[:50]}")
    else:
        image = Image.open(item["image"]).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": "Extract all text exactly as it appears."}
                ]
            }
        ]

        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=512, do_sample=False)

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs['input_ids'], output)
        ]
        pred = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

    with open(item["label"], "r") as f:
        gt = f.read()

    predictions.append({"pred": pred, "gt": gt, "group": item["group"]})

print("Done:", len(predictions))

TESSERACT: receipt_000.png → Invoice no: 79587541
Date of issue: 08/02/2014
Sel
TESSERACT: receipt_001.png → Invoice no: 27216737
Date of issue: 05/23/2018
Sel
TESSERACT: receipt_002.png → Invoice no: 58097062
Date of issue: 04/05/2013
Sel
TESSERACT: receipt_003.png → Invoice no: 28590433
Date of issue: 09/27/2016
Sel
TESSERACT: receipt_004.png → Invoice no: 45802970
Date of issue: 03/30/2021
Sel
TESSERACT: receipt_005.png → Invoice no: 54533510
Date of issue: 02/10/2021
Sel
Done: 36


In [16]:
import re
from jiwer import cer, wer

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def compute_cer(pred, ref):
    return min(cer(normalize_text(ref), normalize_text(pred)), 1.0)

def compute_wer(pred, ref):
    return min(wer(normalize_text(ref), normalize_text(pred)), 1.0)

def compute_accuracy(pred, ref):
    c = compute_cer(pred, ref)
    w = compute_wer(pred, ref)
    return max(0.0, (1.0 - ((c + w) / 2.0)) * 100)

In [17]:
total = 0

for item in predictions:
    total += compute_accuracy(item["pred"], item["gt"])

overall_accuracy = total / len(predictions)

print("Overall Accuracy:", round(overall_accuracy, 2))

Overall Accuracy: 73.91


In [18]:
for p in predictions:
    if p["group"] == "receipts":
        acc = compute_accuracy(p["pred"], p["gt"])
        print(f"acc: {round(acc,2)} | pred: {p['pred'][:60]}")

acc: 94.66 | pred: Invoice no: 79587541
Date of issue: 08/02/2014
Seller: Clien
acc: 93.91 | pred: Invoice no: 27216737
Date of issue: 05/23/2018
Seller: Clien
acc: 82.92 | pred: Invoice no: 58097062
Date of issue: 04/05/2013
Seller: Clien
acc: 82.59 | pred: Invoice no: 28590433
Date of issue: 09/27/2016
Seller: Clien
acc: 86.84 | pred: Invoice no: 45802970
Date of issue: 03/30/2021
Seller: Clien
acc: 84.91 | pred: Invoice no: 54533510
Date of issue: 02/10/2021
Seller: Clien


In [19]:
from collections import defaultdict

group_scores = defaultdict(list)

for item in predictions:
    acc = compute_accuracy(item["pred"], item["gt"])
    group_scores[item["group"]].append(acc)

for group in group_scores:
    avg = sum(group_scores[group]) / len(group_scores[group])
    print(f"{group}: {round(avg, 2)}")

degraded: 41.06
dense_text: 67.63
handwritten: 92.75
printed: 54.35
receipts: 87.64
scene_text: 100.0
